In [1]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory
from oat.model_module.model import OAT
import torch

In [2]:
from torch.utils.data import DataLoader
from oat.data_module.wind_dataset import QueryBatch
data_config = OmegaConf.load('/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/wind_data.yaml')
dataset = factory.create_dataset(data_config)
#train_dataloader, val_dataloader, test_dataloader = factory.create_dataset(data_config)
#batch = next(iter(train_dataloader))

loader = DataLoader(dataset, batch_size=16, shuffle=False)


/scratch/mch/cdonner/observation_assimilation_transformer/.venv/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [3]:
batch = next(iter(loader))

In [4]:
obs_data = batch['observation_data']
num_stations = obs_data['obs'].shape[1]
key_stations = 100
query = QueryBatch(
    coords=obs_data['coords'][:, key_stations:, :],
    topo=obs_data['topo'][:, key_stations:, :],
    geo=obs_data['geo'][:, key_stations:, :],
    obs=obs_data['obs'][:, key_stations:, :],
    mask_obs=obs_data['mask_obs'][:, key_stations:, :],
)
observation_data = {
    'coords': obs_data['coords'][:, :key_stations, :],
    'topo': obs_data['topo'][:, :key_stations, :],
    'geo': obs_data['geo'][:, :key_stations, :],
    'obs': obs_data['obs'][:, :key_stations, :],
    'mask_obs': obs_data['mask_obs'][:, :key_stations, :],
}
batch['observation_data'] = observation_data
batch['query_data'] = query

In [5]:
model = OAT()

In [6]:
with torch.no_grad():
    output = model(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], )

In [8]:

#with torch.no_grad():
# get gradients
pred, loss = model.forward_with_loss(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], target_obs=batch['query_data']['obs'], target_mask_obs=batch['query_data']['mask_obs'])
loss.backward()

In [9]:
for param in model.parameters():
    if param.grad is not None:
        print(param.grad.abs().sum())

tensor(1.7225)
tensor(0.0060)
tensor(0.0007)
tensor(0.0010)
tensor(2.4682e-05)
tensor(0.0246)
tensor(0.0015)
tensor(0.0206)
tensor(0.0023)
tensor(0.0004)
tensor(0.0005)
tensor(0.0247)
tensor(0.0014)
tensor(0.0510)
tensor(0.0024)
tensor(0.0006)
tensor(0.0006)
tensor(3.2375e-05)
tensor(0.0274)
tensor(0.0016)
tensor(0.0237)
tensor(0.0023)
tensor(0.0003)
tensor(0.0006)
tensor(0.0225)
tensor(0.0013)
tensor(0.0419)
tensor(0.0022)
tensor(3.6338)
tensor(0.0290)
tensor(0.0173)
tensor(0.0034)
tensor(0.0034)
tensor(0.0024)
tensor(0.3655)
tensor(0.0069)
tensor(0.3487)
tensor(0.0107)
tensor(0.0028)
tensor(0.0032)
tensor(0.4850)
tensor(0.0083)
tensor(0.9003)
tensor(0.0092)
tensor(0.0028)
tensor(0.0027)
tensor(0.0021)
tensor(0.2498)
tensor(0.0048)
tensor(0.2534)
tensor(0.0071)
tensor(0.0035)
tensor(0.0020)
tensor(0.4053)
tensor(0.0066)
tensor(0.6942)
tensor(0.0069)
tensor(5.3138)
tensor(0.0229)
tensor(0.0145)
tensor(0.0035)
tensor(0.0035)
tensor(0.0013)
tensor(0.5299)
tensor(0.0063)
tensor(0.4977)
te

In [ ]:
output.shape

In [ ]:
torch.sum(batch['query_data']['obs'] == torch.nan)

In [ ]:
torch.sum(torch.any(pred['val_pred'] == torch.nan))

In [ ]:
# check if nans in output
torch.sum(output['val_pred'] == torch.nan)

In [ ]:
torch.arange(10)[]

In [ ]:
batch['observation_data']['mask_obs'][2]